# RAG con ChromaDB: persistir e inspeccionar

ChromaDB no cambia el corpus ni el embedding del notebook anterior: guarda los mismos vectores, textos y metadatos, y automatiza la búsqueda persistente.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from laundry_rag.ingestion import chunk_pages, ingest_manuals
from laundry_rag.paths import CHROMA_DIR
from laundry_rag.retrieval import embed_texts, search_manual
from laundry_rag.vectorstore import COLLECTION_NAME, ChromaManualStore


## 1. Mismo corpus, misma segmentación

La reconstrucción borra y crea **sólo** `manuales_lavadora`; así el resultado es repetible y no duplica documentos.

In [ ]:
pages = ingest_manuals()
chunks = chunk_pages(pages, chunk_words=220, overlap_words=40)
store = ChromaManualStore()
store.rebuild(chunks)
print(f'Ruta persistente: {CHROMA_DIR}')
print(f'Colección: {COLLECTION_NAME}')
print(f'Chunks indexados: {store.count}')


## 2. Ver lo que se guardó

`peek` permite comprobar textos, IDs y metadatos. Los embeddings se almacenan como vectores numéricos; no conviene imprimirlos completos, pero sí su dimensión.

In [ ]:
muestra = store.collection.peek(limit=3)
pd.DataFrame({'id': muestra['ids'], 'documento': muestra['documents'], 'metadatos': muestra['metadatas']})


In [ ]:
registro = store.collection.get(limit=1, include=['embeddings', 'documents', 'metadatas'])
print('Dimensión del embedding:', len(registro['embeddings'][0]))
print('Metadatos:', registro['metadatas'][0])


## 3. Buscar y mostrar distancia

La distancia de Chroma (menor es mejor) aparece junto con texto y trazabilidad para que la respuesta pueda citarse.

In [ ]:
pregunta = '¿Qué debo revisar antes de instalar la lavadora?'
resultados_chroma = store.search(pregunta, top_k=4)
pd.DataFrame([item.to_dict() for item in resultados_chroma])[['manual', 'page', 'section', 'distance', 'text']]


## 4. Comparación con la búsqueda manual

Ambas rutas usan el mismo modelo y chunks. Las diferencias deberían ser de representación de distancia, no de evidencia relevante.

In [ ]:
embeddings = embed_texts([chunk.text for chunk in chunks])
resultados_manual = search_manual(pregunta, chunks, embeddings, top_k=4)
comparacion = pd.DataFrame({
    'manual': [f'{x.manual} p. {x.page}' for x in resultados_manual],
    'distancia_manual': [x.distance for x in resultados_manual],
    'chroma': [f'{x.manual} p. {x.page}' for x in resultados_chroma],
    'distancia_chroma': [x.distance for x in resultados_chroma],
})
comparacion


## Ejercicios

- Ejecuta de nuevo la celda de reconstrucción: el contador debe permanecer igual.
- Usa `collection.get(where={'model': '8MWTW1989'})` para inspeccionar un modelo.
- Cambia `top_k` y discute el equilibrio entre contexto suficiente y ruido.
- Después ejecuta `uv run streamlit run chatbot/app.py`: el chatbot reutiliza esta colección persistente.